In [2]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
    .master("local[4]")
    .appName("tp")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.host", "localhost")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate())

print(spark.sparkContext.uiWebUrl)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/03 11:38:09 WARN Utils: Your hostname, oualid-Latitude-E5570, resolves to a loopback address: 127.0.1.1; using 192.168.51.11 instead (on interface wlp1s0)
26/10/03 11:38:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/03 11:38:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


http://localhost:4040


In [4]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
    .master("local[4]")                            # 1 machine, 4 cores = 4 task slots
    .appName("tp")
    .config("spark.sql.shuffle.partitions", "8")   # 200 is too much for a laptop
    .getOrCreate())

# Customers: 1000 rows
customers = spark.range(1, 1001).select(
    F.col("id").alias("cust_id"),
    F.concat(F.lit("name_"), F.col("id")).alias("name"),
    F.element_at(F.array(F.lit("MA"), F.lit("FR"), F.lit("US"), F.lit("DE")),
                 (F.col("id") % 4 + 1).cast("int")).alias("country"))

# Orders: 1,000,000 rows, and half of them belong to customer 1 (a hot key, for the skew lab)
orders = spark.range(1_000_000).select(
    F.col("id").alias("order_id"),
    F.when(F.rand(42) < 0.5, F.lit(1))
     .otherwise((F.rand(7) * 1000).cast("int") + 1).alias("cust_id"),
    (F.rand(1) * 500).alias("amount"))

# Employees: 5000 rows, 10 departments
employees = spark.range(5000).select(
    F.col("id").alias("emp_id"),
    F.concat(F.lit("emp_"), F.col("id")).alias("name"),
    (F.col("id") % 10).alias("dept_id"),
    (F.rand(3) * 8000 + 2000).cast("int").alias("salary"),
    F.date_add(F.lit("2015-01-01"), (F.rand(5) * 3000).cast("int")).alias("hire_date"))

# Save as Parquet so every lab reads the same data
customers.write.mode("overwrite").parquet("data/customers")
orders.write.mode("overwrite").parquet("data/orders")
employees.write.mode("overwrite").parquet("data/employees")

In [3]:
orders = spark.read.parquet("data/orders")
print(orders.count())
print(orders.rdd.getNumPartitions())

1000000
4


In [4]:
orders = spark.read.parquet("data/orders")            # (a)
big = orders.filter(F.col("amount") > 400)            # (b)
agg = big.groupBy("cust_id").count()                  # (c)
agg.show(5)                                           # (d)
agg.count()                                           # (e)

+-------+-----+
|cust_id|count|
+-------+-----+
|    171|   98|
|     67|  113|
|    536|   97|
|    734|   94|
|    107|   96|
+-------+-----+
only showing top 5 rows


1000

In [7]:
spark.sparkContext.setJobDescription("LAB1 show(5)")
agg.show(5)

spark.sparkContext.setJobDescription("LAB1 count")
agg.count()

+-------+-----+
|cust_id|count|
+-------+-----+
|    171|   98|
|     67|  113|
|    536|   97|
|    734|   94|
|    107|   96|
+-------+-----+
only showing top 5 rows


1000

In [8]:
big = spark.read.parquet("data/orders").filter(F.col("amount") > 400).select("cust_id", "amount")
big.explain(True)

== Parsed Logical Plan ==
'Project ['cust_id, 'amount]
+- Filter (amount#75 > cast(400 as double))
   +- Relation [order_id#73L,cust_id#74,amount#75] parquet

== Analyzed Logical Plan ==
cust_id: int, amount: double
Project [cust_id#74, amount#75]
+- Filter (amount#75 > cast(400 as double))
   +- Relation [order_id#73L,cust_id#74,amount#75] parquet

== Optimized Logical Plan ==
Project [cust_id#74, amount#75]
+- Filter (isnotnull(amount#75) AND (amount#75 > 400.0))
   +- Relation [order_id#73L,cust_id#74,amount#75] parquet

== Physical Plan ==
*(1) Filter (isnotnull(amount#75) AND (amount#75 > 400.0))
+- *(1) ColumnarToRow
   +- FileScan parquet [cust_id#74,amount#75] Batched: true, DataFilters: [isnotnull(amount#75), (amount#75 > 400.0)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/oualid/Desktop/spark/data/orders], PartitionFilters: [], PushedFilters: [IsNotNull(amount), GreaterThan(amount,400.0)], ReadSchema: struct<cust_id:int,amount:double>

#
# A fatal error 

In [5]:
print(orders.rdd.getNumPartitions())                    # orders → 4
print(orders.repartition(10).rdd.getNumPartitions())    # orders → repartition(10)
print(orders.coalesce(2).rdd.getNumPartitions())        # orders → coalesce(2)
print(orders.coalesce(10).rdd.getNumPartitions())       # orders → coalesce(10)

4


[Stage 14:>                                                         (0 + 4) / 4]

10
2
4


In [6]:
orders = spark.read.parquet("data/orders")
customers = spark.read.parquet("data/customers")

j1 = orders.join(customers, "cust_id")
j1.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [cust_id#35, order_id#34L, amount#36, name#38, country#39]
   +- BroadcastHashJoin [cast(cust_id#35 as bigint)], [cust_id#37L], Inner, BuildRight, false, false
      :- Filter isnotnull(cust_id#35)
      :  +- FileScan parquet [order_id#34L,cust_id#35,amount#36] Batched: true, DataFilters: [isnotnull(cust_id#35)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/oualid/Desktop/spark/data/orders], PartitionFilters: [], PushedFilters: [IsNotNull(cust_id)], ReadSchema: struct<order_id:bigint,cust_id:int,amount:double>
      +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, bigint, false]),false), [plan_id=299]
         +- Filter isnotnull(cust_id#37L)
            +- FileScan parquet [cust_id#37L,name#38,country#39] Batched: true, DataFilters: [isnotnull(cust_id#37L)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/oualid/Desktop/spark/data/customers], PartitionFilters: [], Pus

In [7]:
j2 = orders.join(customers.hint("merge"), "cust_id")
j2.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [cust_id#35, order_id#34L, amount#36, name#38, country#39]
   +- SortMergeJoin [cast(cust_id#35 as bigint)], [cust_id#37L], Inner
      :- Sort [cast(cust_id#35 as bigint) ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(cast(cust_id#35 as bigint), 8), ENSURE_REQUIREMENTS, [plan_id=329]
      :     +- Filter isnotnull(cust_id#35)
      :        +- FileScan parquet [order_id#34L,cust_id#35,amount#36] Batched: true, DataFilters: [isnotnull(cust_id#35)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/oualid/Desktop/spark/data/orders], PartitionFilters: [], PushedFilters: [IsNotNull(cust_id)], ReadSchema: struct<order_id:bigint,cust_id:int,amount:double>
      +- Sort [cust_id#37L ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(cust_id#37L, 8), ENSURE_REQUIREMENTS, [plan_id=330]
            +- Filter isnotnull(cust_id#37L)
               +- FileScan parquet [cust_id#37L,nam

In [8]:
spark.sparkContext.setJobDescription("LAB4 broadcast")
j1.count()

spark.sparkContext.setJobDescription("LAB4 sortmerge")
j2.count()

1000000